# NurtureJoy Two-Stage Emotion Classifier — BERT (bert-base-uncased)

This notebook trains a **two-stage classifier**:

- **Stage 1:** POSITIVE / NEUTRAL / NEGATIVE  
- **Stage 2:** (only if NEGATIVE) STRESS / ANXIETY / LOW_MOOD / HIGH_DISTRESS

It then evaluates both stages and saves artifacts for deployment.

**Dataset expected columns:** `text`, `category`  
Default path: `./nurturejoy_emotion_with_preg_hardboundary.csv`

> Tip: Run on GPU if available for transformer notebooks.


In [ ]:

import os, re, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

EMOTION_PATH = os.environ.get("EMOTION_PATH", "./nurturejoy_emotion_with_preg_hardboundary.csv")
assert os.path.exists(EMOTION_PATH), f"Missing dataset at: {EMOTION_PATH}"

def clean_text(t: str) -> str:
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t

LABELS6 = ["POSITIVE","NEUTRAL","ANXIETY","STRESS","LOW_MOOD","HIGH_DISTRESS"]
NEG4 = ["STRESS","ANXIETY","LOW_MOOD","HIGH_DISTRESS"]

df = pd.read_csv(EMOTION_PATH).dropna(subset=["text","category"]).copy()
df["text"] = df["text"].map(clean_text)
df = df[df["text"].str.len() >= 5]
df = df[df["category"].isin(LABELS6)].copy()

def to_stage1(lbl):
    if lbl == "POSITIVE": return "POSITIVE"
    if lbl == "NEUTRAL":  return "NEUTRAL"
    return "NEGATIVE"

df["stage1"] = df["category"].map(to_stage1)
df["stage2"] = df["category"].where(df["stage1"]=="NEGATIVE", other="NA")

print("Rows:", len(df))
print("Stage1 distribution:\n", df["stage1"].value_counts())
print("Stage2 (neg) distribution:\n", df[df.stage1=="NEGATIVE"]["category"].value_counts())
df.head(3)


In [ ]:
# Two-stage transformer training


In [ ]:

# If needed (fresh env / Colab):
# !pip -q install transformers datasets accelerate torch scikit-learn pandas numpy

import os, numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
set_seed(42)
print("CUDA available:", torch.cuda.is_available())
MODEL_CKPT = "bert-base-uncased"


In [ ]:

# Split once (stage1 stratified)
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df["stage1"])
val_df, test_df = train_test_split(test_df, test_size=0.50, random_state=42, stratify=test_df["stage1"])
print("Sizes:", len(train_df), len(val_df), len(test_df))


In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds==labels).mean())}


In [ ]:

# Stage 1 labels
stage1_labels = ["POSITIVE","NEUTRAL","NEGATIVE"]
s1_label2id = {l:i for i,l in enumerate(stage1_labels)}
s1_id2label = {i:l for l,i in s1_label2id.items()}

train_s1 = train_df.copy()
val_s1   = val_df.copy()
test_s1  = test_df.copy()
train_s1["label"] = train_s1["stage1"].map(s1_label2id)
val_s1["label"]   = val_s1["stage1"].map(s1_label2id)
test_s1["label"]  = test_s1["stage1"].map(s1_label2id)

ds_train_s1 = Dataset.from_pandas(train_s1[["text","label"]].reset_index(drop=True))
ds_val_s1   = Dataset.from_pandas(val_s1[["text","label"]].reset_index(drop=True))
ds_test_s1  = Dataset.from_pandas(test_s1[["text","label"]].reset_index(drop=True))

tokenizer_s1 = AutoTokenizer.from_pretrained(MODEL_CKPT)
def tok_s1(batch): return tokenizer_s1(batch["text"], truncation=True)

ds_train_s1 = ds_train_s1.map(tok_s1, batched=True)
ds_val_s1   = ds_val_s1.map(tok_s1, batched=True)
ds_test_s1  = ds_test_s1.map(tok_s1, batched=True)

collator_s1 = DataCollatorWithPadding(tokenizer=tokenizer_s1)


In [ ]:

model_s1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(stage1_labels),
    id2label=s1_id2label,
    label2id=s1_label2id
)

args_s1 = TrainingArguments(
    output_dir="artifacts_bert/stage1",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer_s1 = Trainer(
    model=model_s1,
    args=args_s1,
    train_dataset=ds_train_s1,
    eval_dataset=ds_val_s1,
    tokenizer=tokenizer_s1,
    data_collator=collator_s1,
    compute_metrics=compute_metrics
)
trainer_s1.train()


In [ ]:

pred_s1 = trainer_s1.predict(ds_test_s1)
y_true = pred_s1.label_ids
y_pred = np.argmax(pred_s1.predictions, axis=-1)

from sklearn.metrics import classification_report, confusion_matrix
print("=== STAGE 1 TEST REPORT ===")
print(classification_report(y_true, y_pred, target_names=stage1_labels))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


In [ ]:

# Stage 2 dataset: NEGATIVE rows only
train_neg = train_df[train_df["stage1"]=="NEGATIVE"].copy()
val_neg   = val_df[val_df["stage1"]=="NEGATIVE"].copy()
test_neg  = test_df[test_df["stage1"]=="NEGATIVE"].copy()

stage2_labels = ["STRESS","ANXIETY","LOW_MOOD","HIGH_DISTRESS"]
s2_label2id = {l:i for i,l in enumerate(stage2_labels)}
s2_id2label = {i:l for l,i in s2_label2id.items()}

train_neg = train_neg[train_neg["category"].isin(stage2_labels)]
val_neg   = val_neg[val_neg["category"].isin(stage2_labels)]
test_neg  = test_neg[test_neg["category"].isin(stage2_labels)]

train_neg["label"] = train_neg["category"].map(s2_label2id)
val_neg["label"]   = val_neg["category"].map(s2_label2id)
test_neg["label"]  = test_neg["category"].map(s2_label2id)

ds_train_s2 = Dataset.from_pandas(train_neg[["text","label"]].reset_index(drop=True))
ds_val_s2   = Dataset.from_pandas(val_neg[["text","label"]].reset_index(drop=True))
ds_test_s2  = Dataset.from_pandas(test_neg[["text","label"]].reset_index(drop=True))

tokenizer_s2 = AutoTokenizer.from_pretrained(MODEL_CKPT)
def tok_s2(batch): return tokenizer_s2(batch["text"], truncation=True)

ds_train_s2 = ds_train_s2.map(tok_s2, batched=True)
ds_val_s2   = ds_val_s2.map(tok_s2, batched=True)
ds_test_s2  = ds_test_s2.map(tok_s2, batched=True)

collator_s2 = DataCollatorWithPadding(tokenizer=tokenizer_s2)


In [ ]:

model_s2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(stage2_labels),
    id2label=s2_id2label,
    label2id=s2_label2id
)

args_s2 = TrainingArguments(
    output_dir="artifacts_bert/stage2",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer_s2 = Trainer(
    model=model_s2,
    args=args_s2,
    train_dataset=ds_train_s2,
    eval_dataset=ds_val_s2,
    tokenizer=tokenizer_s2,
    data_collator=collator_s2,
    compute_metrics=compute_metrics
)
trainer_s2.train()


In [ ]:

pred_s2 = trainer_s2.predict(ds_test_s2)
y_true2 = pred_s2.label_ids
y_pred2 = np.argmax(pred_s2.predictions, axis=-1)

from sklearn.metrics import classification_report, confusion_matrix
print("=== STAGE 2 TEST REPORT ===")
print(classification_report(y_true2, y_pred2, target_names=stage2_labels))
print("Confusion matrix:\n", confusion_matrix(y_true2, y_pred2))


In [ ]:

# Save models/tokenizers + a joblib "bundle" for easy loading later
import joblib, os

os.makedirs("artifacts_bert/bundle", exist_ok=True)

trainer_s1.save_model("artifacts_bert/bundle/stage1_model")
tokenizer_s1.save_pretrained("artifacts_bert/bundle/stage1_model")

trainer_s2.save_model("artifacts_bert/bundle/stage2_model")
tokenizer_s2.save_pretrained("artifacts_bert/bundle/stage2_model")

bundle = {
    "framework": "transformers",
    "model_checkpoint": MODEL_CKPT,
    "stage1_model_dir": "artifacts_bert/bundle/stage1_model",
    "stage2_model_dir": "artifacts_bert/bundle/stage2_model",
    "stage1_labels": stage1_labels,
    "stage2_labels": stage2_labels
}

joblib.dump(bundle, "artifacts_bert/bundle/two_stage_bundle.joblib")
print("Saved bundle joblib at: artifacts_bert/bundle/two_stage_bundle.joblib")


In [ ]:

# Simple inference check
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

def load_stage(model_dir):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir)
    mdl.eval()
    return tok, mdl

tok1, mdl1 = load_stage(bundle["stage1_model_dir"])
tok2, mdl2 = load_stage(bundle["stage2_model_dir"])

@torch.no_grad()
def predict_two_stage_transformer(text: str):
    text = clean_text(text)

    inp = tok1(text, return_tensors="pt", truncation=True)
    out = mdl1(**inp)
    s1_id = int(out.logits.argmax(dim=-1).item())
    s1_lbl = bundle["stage1_labels"][s1_id]
    if s1_lbl in ["POSITIVE","NEUTRAL"]:
        return s1_lbl

    inp2 = tok2(text, return_tensors="pt", truncation=True)
    out2 = mdl2(**inp2)
    s2_id = int(out2.logits.argmax(dim=-1).item())
    return bundle["stage2_labels"][s2_id]

tests = [
    "I’m excited today after my ultrasound!",
    "Today feels normal, just tired.",
    "I’m overwhelmed with work and appointments, it’s too much to handle.",
    "I can’t stop worrying about my glucose test results.",
    "I feel sad and lonely and I keep crying.",
    "I feel hopeless and unsafe with my thoughts."
]
for t in tests:
    print(predict_two_stage_transformer(t), " | ", t)
